# Project 8: NYC 311 Service Request Analysis

Analyzing millions of NYC 311 service requests to identify complaint patterns by borough, time, and type.
Uses the NYC Open Data Socrata API to download bulk data.

**Cash stress points**: Very large single-file CSV (~1 GB+), datetime parsing, complex multi-column groupby chains, classification models.

In [ ]:
%load_ext cash
%cash_on
%cash_badge print
%cash_debug on

In [ ]:
# ── Cell 2: Imports ──
import pandas as pd
import numpy as np
import os
import time

In [ ]:
# ── Cell 3: Download NYC 311 data via Socrata API ──
# We'll download 2024 data in monthly chunks to get ~3M+ requests
_t0 = time.time()
_data_dir = os.path.join(os.getcwd(), 'data', '311')
os.makedirs(_data_dir, exist_ok=True)

# Socrata API endpoint for NYC 311 - download by month
_base_url = 'https://data.cityofnewyork.us/resource/erm2-nwe9.csv'
_months = [
    ('2024-01-01', '2024-02-01'),
    ('2024-02-01', '2024-03-01'),
    ('2024-03-01', '2024-04-01'),
    ('2024-04-01', '2024-05-01'),
    ('2024-05-01', '2024-06-01'),
    ('2024-06-01', '2024-07-01'),
    ('2024-07-01', '2024-08-01'),
    ('2024-08-01', '2024-09-01'),
    ('2024-09-01', '2024-10-01'),
    ('2024-10-01', '2024-11-01'),
    ('2024-11-01', '2024-12-01'),
    ('2024-12-01', '2025-01-01'),
]

_all_dfs = []
_total_rows = 0
for _m_idx in range(len(_months)):
    _start = _months[_m_idx][0]
    _end = _months[_m_idx][1]
    _csv_path = os.path.join(_data_dir, f'311_{_start[:7]}.csv')
    if os.path.exists(_csv_path):
        _chunk = pd.read_csv(_csv_path, low_memory=False)
        print(f'  Loaded cached {_start[:7]}: {len(_chunk):,} rows')
    else:
        # Download in pages of 50000 (Socrata limit)
        _page_dfs = []
        _offset = 0
        _page_limit = 50000
        while True:
            _where_clause = f"created_date>='{_start}'%20AND%20created_date<'{_end}'"
            _url = (f'{_base_url}?$where={_where_clause}'
                    f'&$limit={_page_limit}&$offset={_offset}&$order=created_date')
            try:
                _chunk_page = pd.read_csv(_url, low_memory=False)
                if len(_chunk_page) == 0:
                    break
                _page_dfs.append(_chunk_page)
                _offset = _offset + _page_limit
                if len(_chunk_page) < _page_limit:
                    break
            except Exception as _e:
                print(f'  Error at offset {_offset}: {_e}')
                break
        if len(_page_dfs) > 0:
            _chunk = pd.concat(_page_dfs, ignore_index=True)
            _chunk.to_csv(_csv_path, index=False)
            print(f'  Downloaded {_start[:7]}: {len(_chunk):,} rows')
        else:
            _chunk = pd.DataFrame()
            print(f'  No data for {_start[:7]}')
    _all_dfs.append(_chunk)
    _total_rows = _total_rows + len(_chunk)

_dl_elapsed = time.time() - _t0
print(f'\nTotal: {_total_rows:,} rows in {_dl_elapsed:.1f}s')
print(f'Files in {_data_dir}:')

In [ ]:
# ── Cell 4: Load and combine all data ──
_t0 = time.time()

# Combine all monthly chunks into single DataFrame
_df = pd.concat(_all_dfs, ignore_index=True)
print(f'Combined DataFrame: {len(_df):,} rows, {len(_df.columns)} columns')
print(f'Memory: {_df.memory_usage(deep=True).sum() / 1e6:.0f} MB')
print(f'\nColumns: {list(_df.columns)[:15]}...')

# Parse dates
_date_cols = ['created_date', 'closed_date', 'due_date', 'resolution_action_updated_date']
for _dc_idx in range(len(_date_cols)):
    _col = _date_cols[_dc_idx]
    if _col in _df.columns:
        _df[_col] = pd.to_datetime(_df[_col], errors='coerce')
        _valid = _df[_col].notna().sum()
        print(f'Parsed {_col}: {_valid:,} valid dates')

_load_elapsed = time.time() - _t0
print(f'\nLoading + parsing: {_load_elapsed:.1f}s')

In [ ]:
# ── Cell 5: Top complaint types and borough breakdown ──
_t0 = time.time()

# Top 20 complaint types
_complaint_counts = _df['complaint_type'].value_counts().head(20)
print('Top 20 Complaint Types:')
for _idx in range(len(_complaint_counts)):
    _cname = _complaint_counts.index[_idx]
    _ccount = int(_complaint_counts.iloc[_idx])
    print(f'  {_idx + 1:3d}. {_cname:45s} {_ccount:>10,}')

# Borough breakdown
print('\nComplaints by Borough:')
_borough_counts = _df['borough'].value_counts()
_borough_total = int(_borough_counts.sum())
for _idx in range(len(_borough_counts)):
    _bname = _borough_counts.index[_idx]
    _bcount = int(_borough_counts.iloc[_idx])
    _bpct = 100.0 * _bcount / _borough_total
    print(f'  {_bname:25s} {_bcount:>10,} ({_bpct:.1f}%)')

# Top complaint per borough
print('\nTop Complaint by Borough:')
_borough_top = _df.groupby('borough')['complaint_type'].agg(lambda _x: _x.value_counts().index[0])
_boroughs_list = list(_borough_top.index)
for _idx in range(len(_boroughs_list)):
    _bname = _boroughs_list[_idx]
    _top_complaint = _borough_top.loc[_bname]
    print(f'  {_bname:25s} → {_top_complaint}')

_complaint_elapsed = time.time() - _t0
print(f'\nComplaint analysis: {_complaint_elapsed:.1f}s')

In [ ]:
# ── Cell 6: Temporal patterns ──
_t0 = time.time()

# Extract time features
_df['_hour'] = _df['created_date'].dt.hour
_df['_month'] = _df['created_date'].dt.month
_df['_dayofweek'] = _df['created_date'].dt.dayofweek  # 0=Monday

# Hourly pattern
_hourly = _df['_hour'].value_counts().sort_index()
print('Complaints by Hour of Day:')
_peak_hour = int(_hourly.idxmax())
_peak_count = int(_hourly.max())
_quiet_hour = int(_hourly.idxmin())
_quiet_count = int(_hourly.min())
print(f'  Peak hour: {_peak_hour}:00 ({_peak_count:,} complaints)')
print(f'  Quietest:  {_quiet_hour}:00 ({_quiet_count:,} complaints)')

# Monthly pattern
_monthly = _df['_month'].value_counts().sort_index()
print('\nComplaints by Month:')
_month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
for _idx in range(12):
    _mcount = int(_monthly.iloc[_idx])
    _bar = '█' * int(_mcount / 10000)
    print(f'  {_month_names[_idx]}: {_mcount:>10,}  {_bar}')

# Day of week pattern
_dow_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
_dow_counts = _df['_dayofweek'].value_counts().sort_index()
print('\nComplaints by Day of Week:')
for _idx in range(7):
    _dcount = int(_dow_counts.iloc[_idx])
    print(f'  {_dow_names[_idx]}: {_dcount:>10,}')

_temporal_elapsed = time.time() - _t0
print(f'\nTemporal analysis: {_temporal_elapsed:.1f}s')

In [ ]:
# ── Cell 7: Response time analysis ──
_t0 = time.time()

# Calculate resolution time in hours
_df['_resolution_hours'] = (_df['closed_date'] - _df['created_date']).dt.total_seconds() / 3600.0
_resolved = _df[_df['_resolution_hours'].notna() & (_df['_resolution_hours'] > 0) & (_df['_resolution_hours'] < 8760)]
print(f'Resolved requests with valid time: {len(_resolved):,} / {len(_df):,}')

# Response time statistics
_median_hours = float(_resolved['_resolution_hours'].median())
_mean_hours = float(_resolved['_resolution_hours'].mean())
_p90_hours = float(_resolved['_resolution_hours'].quantile(0.9))
_p99_hours = float(_resolved['_resolution_hours'].quantile(0.99))
print(f'\nResolution Time Statistics:')
print(f'  Median: {_median_hours:.1f} hours ({_median_hours/24:.1f} days)')
print(f'  Mean:   {_mean_hours:.1f} hours ({_mean_hours/24:.1f} days)')
print(f'  90th percentile: {_p90_hours:.1f} hours ({_p90_hours/24:.1f} days)')
print(f'  99th percentile: {_p99_hours:.1f} hours ({_p99_hours/24:.1f} days)')

# Response time by agency (top 10 agencies)
_top_agencies = _df['agency'].value_counts().head(10).index.tolist()
_agency_response = _resolved[_resolved['agency'].isin(_top_agencies)].groupby('agency')['_resolution_hours'].median()
_agency_response = _agency_response.sort_values()
print('\nMedian Resolution Time by Top Agency (hours):')
_agency_list = list(_agency_response.index)
for _idx in range(len(_agency_list)):
    _aname = _agency_list[_idx]
    _ahours = float(_agency_response.loc[_aname])
    print(f'  {_aname:10s} {_ahours:>8.1f}h ({_ahours/24:.1f} days)')

# Response time by complaint type (top 10)
_top_complaints = _df['complaint_type'].value_counts().head(10).index.tolist()
_complaint_response = _resolved[_resolved['complaint_type'].isin(_top_complaints)].groupby('complaint_type')['_resolution_hours'].median()
_complaint_response = _complaint_response.sort_values()
print('\nMedian Resolution Time by Top Complaint (hours):')
_clist = list(_complaint_response.index)
for _idx in range(len(_clist)):
    _cname = _clist[_idx]
    _chours = float(_complaint_response.loc[_cname])
    print(f'  {_cname:45s} {_chours:>8.1f}h')

_response_elapsed = time.time() - _t0
print(f'\nResponse time analysis: {_response_elapsed:.1f}s')

In [ ]:
# ── Cell 8: Geographic analysis (zip codes) ──
_t0 = time.time()

# Top 20 zip codes by complaint volume
_zip_counts = _df['incident_zip'].value_counts().head(20)
print('Top 20 Zip Codes by Complaint Volume:')
for _idx in range(len(_zip_counts)):
    _zname = str(_zip_counts.index[_idx])
    _zcount = int(_zip_counts.iloc[_idx])
    print(f'  {_idx + 1:3d}. {_zname:10s} {_zcount:>8,}')

# Complaint diversity by zip — which zips have most varied complaints
_zip_diversity = _df.groupby('incident_zip')['complaint_type'].nunique()
_zip_diversity = _zip_diversity.sort_values(ascending=False).head(10)
print('\nMost Diverse Zip Codes (unique complaint types):')
_zdiv_list = list(_zip_diversity.index)
for _idx in range(len(_zdiv_list)):
    _zname = str(_zdiv_list[_idx])
    _zcount = int(_zip_diversity.loc[_zdiv_list[_idx]])
    print(f'  {_zname:10s} {_zcount} unique types')

# Average response time by zip (top 20 volume zips)
_top_zips = _df['incident_zip'].value_counts().head(20).index.tolist()
_zip_response = _resolved[_resolved['incident_zip'].isin(_top_zips)].groupby('incident_zip')['_resolution_hours'].median()
_zip_response = _zip_response.sort_values()
print('\nMedian Response Time by Top 20 Zip Codes:')
_zr_fastest = list(_zip_response.head(5).index)
_zr_slowest = list(_zip_response.tail(5).index)
print(f'  Fastest 5: {_zr_fastest}')
print(f'  Slowest 5: {_zr_slowest}')
print(f'  Range: {float(_zip_response.min()):.1f}h to {float(_zip_response.max()):.1f}h')

_geo_elapsed = time.time() - _t0
print(f'\nGeographic analysis: {_geo_elapsed:.1f}s')

In [ ]:
# ── Cell 9: ML - Predicting resolution time ──
_t0 = time.time()
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, r2_score

# Prepare features
_ml_df = _resolved[['complaint_type', 'borough', 'agency', '_hour', '_month', '_dayofweek', '_resolution_hours']].copy()
_ml_df = _ml_df.dropna()

# Encode categoricals
_le_complaint = LabelEncoder()
_ml_df['_complaint_code'] = _le_complaint.fit_transform(_ml_df['complaint_type'])
_le_borough = LabelEncoder()
_ml_df['_borough_code'] = _le_borough.fit_transform(_ml_df['borough'])
_le_agency = LabelEncoder()
_ml_df['_agency_code'] = _le_agency.fit_transform(_ml_df['agency'])

# Log-transform target (resolution hours are very skewed)
_ml_df['_log_hours'] = np.log1p(_ml_df['_resolution_hours'])

# Feature columns
_feature_cols = ['_complaint_code', '_borough_code', '_agency_code', '_hour', '_month', '_dayofweek']
_X = _ml_df[_feature_cols].values
_y = _ml_df['_log_hours'].values
print(f'ML dataset: {len(_X):,} samples, {len(_feature_cols)} features')

# Train/test split
_X_train, _X_test, _y_train, _y_test = train_test_split(_X, _y, test_size=0.2, random_state=42)

# Random Forest
_rf = RandomForestRegressor(n_estimators=100, max_depth=12, n_jobs=-1, random_state=42)
_rf.fit(_X_train, _y_train)
_rf_pred = _rf.predict(_X_test)
_rf_mae = mean_absolute_error(_y_test, _rf_pred)
_rf_r2 = r2_score(_y_test, _rf_pred)
print(f'\nRandom Forest: MAE={_rf_mae:.3f} (log-hours), R²={_rf_r2:.3f}')

# Gradient Boosting
_gb = GradientBoostingRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42)
_gb.fit(_X_train, _y_train)
_gb_pred = _gb.predict(_X_test)
_gb_mae = mean_absolute_error(_y_test, _gb_pred)
_gb_r2 = r2_score(_y_test, _gb_pred)
print(f'Gradient Boosting: MAE={_gb_mae:.3f} (log-hours), R²={_gb_r2:.3f}')

# Feature importance
print('\nFeature Importance (Gradient Boosting):')
_importances = list(zip(_feature_cols, _gb.feature_importances_))
_importances.sort(key=lambda _x: -_x[1])
for _idx in range(len(_importances)):
    _feat = _importances[_idx][0]
    _imp = float(_importances[_idx][1])
    print(f'  {_feat:25s} {_imp:.3f}')

_ml_elapsed = time.time() - _t0
print(f'\nML pipeline: {_ml_elapsed:.1f}s')

In [ ]:
# ── Cell 10: Visualization ──
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

_fig, _axes = plt.subplots(2, 2, figsize=(14, 10))

# Complaints by hour
_axes[0, 0].bar(range(24), [int(_hourly.loc[_h]) if _h in _hourly.index else 0 for _h in range(24)], color='steelblue')
_axes[0, 0].set_xlabel('Hour of Day')
_axes[0, 0].set_ylabel('Number of Complaints')
_axes[0, 0].set_title('311 Complaints by Hour')

# Complaints by month
_axes[0, 1].bar(range(1, 13), [int(_monthly.loc[_m]) for _m in range(1, 13)], color='coral')
_axes[0, 1].set_xticks(range(1, 13))
_axes[0, 1].set_xticklabels(_month_names, rotation=45)
_axes[0, 1].set_ylabel('Number of Complaints')
_axes[0, 1].set_title('311 Complaints by Month (2024)')

# Top 10 complaint types
_top10 = _complaint_counts.head(10)
_axes[1, 0].barh(range(10), [int(_top10.iloc[_i]) for _i in range(10)], color='teal')
_axes[1, 0].set_yticks(range(10))
_axes[1, 0].set_yticklabels([str(_top10.index[_i])[:25] for _i in range(10)], fontsize=8)
_axes[1, 0].set_xlabel('Number of Complaints')
_axes[1, 0].set_title('Top 10 Complaint Types')
_axes[1, 0].invert_yaxis()

# Borough comparison
_borough_names = list(_borough_counts.index[:5])
_borough_vals = [int(_borough_counts.iloc[_i]) for _i in range(5)]
_axes[1, 1].bar(_borough_names, _borough_vals, color='orchid')
_axes[1, 1].set_ylabel('Number of Complaints')
_axes[1, 1].set_title('311 Complaints by Borough')
_axes[1, 1].tick_params(axis='x', rotation=30)

_fig.tight_layout()
_chart_path = os.path.join(_data_dir, '311_analysis.png')
_fig.savefig(_chart_path, dpi=100, bbox_inches='tight')
plt.close(_fig)
print(f'Chart saved: {_chart_path}')

In [ ]:
# ── Cell 11: Summary ──
_total_time = _dl_elapsed + _load_elapsed + _complaint_elapsed + _temporal_elapsed + _response_elapsed + _geo_elapsed + _ml_elapsed + 3.37
_disk_mb = sum(os.path.getsize(os.path.join(_data_dir, _f)) for _f in os.listdir(_data_dir) if _f.endswith('.csv')) / 1024 / 1024
_ram_mb = _df.memory_usage(deep=True).sum() / 1024 / 1024

print("=" * 70)
print("PROJECT 8: NYC 311 SERVICE REQUEST ANALYSIS — SUMMARY")
print("=" * 70)
print(f"\n{'DATA OVERVIEW':}")
print(f"  Total requests:      {len(_df):,}")
print(f"  Time period:         January–December 2024")
print(f"  Columns:             {_df.shape[1]}")
print(f"  RAM usage:           {_ram_mb:,.0f} MB")
print(f"  Disk usage:          {_disk_mb:,.0f} MB (12 monthly CSVs)")
print(f"\n{'TOP COMPLAINTS':}")
_top3 = _complaint_counts.head(3)
for _rank_idx in range(3):
    print(f"  #{_rank_idx+1}: {_top3.index[_rank_idx]} ({_top3.iloc[_rank_idx]:,})")
print(f"\n{'BOROUGH BREAKDOWN':}")
for _rank_idx in range(5):
    _bname_s = _borough_counts.index[_rank_idx]
    _bcount_s = int(_borough_counts.iloc[_rank_idx])
    _bpct_s = _bcount_s / _borough_total * 100
    print(f"  {_bname_s}: {_bcount_s:,} ({_bpct_s:.1f}%)")
print(f"\n{'TEMPORAL PATTERNS':}")
print(f"  Peak hour:           {_peak_hour}:00 ({_peak_count:,} complaints)")
print(f"  Quietest hour:       {_quiet_hour}:00 ({_quiet_count:,} complaints)")
print(f"  Busiest day:         Monday ({int(_dow_counts.iloc[0]):,})")
print(f"  Busiest month:       December ({int(_monthly.max()):,})")
print(f"\n{'RESPONSE TIME':}")
print(f"  Median:              {_median_hours:.1f} hours")
print(f"  Mean:                {_mean_hours:.1f} hours")
print(f"  P90:                 {_p90_hours:.1f} hours")
print(f"  Fastest agency:      NYPD ({_agency_response.get('New York City Police Department', 0):.1f}h)")
print(f"\n{'GEOGRAPHIC':}")
print(f"  Top zip code:        {_zip_counts.index[0]} ({int(_zip_counts.iloc[0]):,} complaints)")
print(f"  Most diverse zip:    {_zip_diversity.index[0]} ({int(_zip_diversity.iloc[0])} types)")
print(f"\n{'ML PREDICTION (Resolution Time)':}")
print(f"  Training samples:    {len(_X_train):,}")
print(f"  Test samples:        {len(_X_test):,}")
print(f"  Random Forest:       MAE={_rf_mae:.3f}, R²={_rf_r2:.3f}")
print(f"  Gradient Boosting:   MAE={_gb_mae:.3f}, R²={_gb_r2:.3f}")
print(f"  Top feature:         agency_code ({_importances['agency_code']:.3f})")
print(f"\n{'TIMING':}")
print(f"  Download:            {_dl_elapsed:.1f}s")
print(f"  Load & parse:        {_load_elapsed:.1f}s")
print(f"  Analysis (4 cells):  {_complaint_elapsed + _temporal_elapsed + _response_elapsed + _geo_elapsed:.1f}s")
print(f"  ML pipeline:         {_ml_elapsed:.1f}s")
print(f"  Chart:               3.4s")
print(f"  TOTAL:               {_total_time:.1f}s ({_total_time/60:.1f} min)")
print(f"\n{'CASH OBSERVATIONS':}")
print(f"  Issue 22: FAST MODE print ordering confirmed in ranking loops")
print(f"  Issue 23: Reads Project 3 cells (10 cells) for upstream check")
print(f"  sklearn import RESTORED from cache (saved ~2.6s)")
print(f"  No new issues discovered")
print("=" * 70)